In [1]:
## classification of transitory myeloid cells 
## Logistic regression, XGBoost, RandomForest 

#import cuml.accel
runs = 5 

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

#from sklearn.preprocessing import OneHotEncoder
#from sklearn.preprocessing import LabelBinarizer

print('Reading data ...')
data=pd.read_csv("../MLdata_Myl_LUAD/Merged_myeloid_data.csv")
#print(data)

data_cleaned = data.dropna(subset=['label']) 
print(data_cleaned.shape)


Reading data ...


In [2]:
print('Encoding classes ...')

data_cleaned = data_cleaned.loc[data_cleaned['label'] != 'Myl_DomTNK']
data_cleaned = data_cleaned.loc[data_cleaned['label'] != 'Myl_DomEpi']

mapping = {'Myl_DomMyl': 0, 'TransMyl': 1}
data_cleaned.loc[:,'status_encoded'] = data_cleaned.loc[:,'label'].map(mapping)
#print(data_cleaned)
numClasses = 2

#le = LabelEncoder()
#data_cleaned['status_encoded'] = le.fit_transform(data_cleaned['label']) 
    
#mapping = {label: index for index, label in enumerate(le.classes_)}
#numClasses = len(le.classes_)
#mapping = {label: index for index, label in enumerate(le.classes_)}
#print("Encoded Mapping:", mapping)

print(data_cleaned['status_encoded'].value_counts()) 

mapping = None 

Encoding classes ...
status_encoded
0    4643
1     543
Name: count, dtype: int64


C:\Users\rdhar\AppData\Local\Temp\ipykernel_1364\1311819053.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_cleaned.loc[:,'status_encoded'] = data_cleaned.loc[:,'label'].map(mapping)


In [3]:
## Load
from sklearn.feature_selection import SelectKBest, f_classif, RFECV, SelectFromModel, VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import QuantileTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb

from sklearn.metrics import classification_report, balanced_accuracy_score,accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
import matplotlib.pyplot as plt
import statistics
import shap

scaler = StandardScaler() 
scaler.set_output(transform="pandas")
normalizer = QuantileTransformer(output_distribution='normal', n_quantiles=4)
#xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,tree_method="hist")
#pipeline = Pipeline([scaler, normalizer, feature_selector1, xgb_model])

In [4]:
## =========== XGBoost ================

#clf = RandomForestClassifier(class_weight="balanced")
lrm = LogisticRegression(solver='lbfgs', C=0.1, max_iter=1000)
rfecv = RFECV(estimator=lrm, step=1, cv=StratifiedKFold(5), scoring='roc_auc')

pipeline = Pipeline(
    steps=[
        ("scaling",scaler),
        ("norm",normalizer),
        ("variance", VarianceThreshold()),
        ("feature_sel1",SelectKBest(score_func=f_classif, k=200)),
        ("feature_sel2",SelectFromModel(LogisticRegression(solver='lbfgs', C=0.1, max_iter=1000))),
        ("feature_sel3",rfecv),
        ("xgb_model", xgb.XGBClassifier(tree_method="hist"))
    ]
)

param_grid = {
    "xgb_model__max_depth": [ 4, 6, 8 ], 
    "xgb_model__n_estimators": [ 50, 100, 150], 
    "xgb_model__learning_rate": [ 0.1, 0.05]
}

wrname = '../results_Merged_Myl/Results_MLrun_Merged_LUAD_run_XGB.txt'
wrf = open(wrname,'w')

acc_scores = []
acc_scores_bal = []
f1_scores = []
prec_scores = []
rec_scores = []
#roc_scores = []

nat_imp = {}
perm_imp = {}
shap_imp = {}


print('Sampling data ...')

for k in range(0,runs):
    print(k)
    wrf.write('Round: {}\n'.format(k))

    df1 = data_cleaned.loc[data_cleaned['status_encoded'] == 0]
    df2 = data_cleaned.loc[data_cleaned['status_encoded'] == 1]

    min_val = min(len(df1),len(df2))
  
    df1_sample = df1.sample(n=min_val)
    df1 = None 
    df2_sample = df2.sample(n=min_val)
    df2 = None 
 
    final_df = pd.concat([df1_sample, df2_sample], axis=0)
    df1_sample = None 
    df2_sample = None 

    
    print(final_df['status_encoded'].value_counts()) 

    xnames=final_df.columns[1:len(final_df.columns)-2]
    #print(xnames)
    X=final_df[xnames]
    y=final_df.status_encoded
    
    xnames = None 
    final_df = None  

    train_X, test_X, train_y, test_y = train_test_split(X, y, stratify=y, train_size=0.75)

    #sample_weights = compute_sample_weight(
    #    class_weight='balanced',
    #    y = train_y #provide your own target name
    #)

    #train_X_embedded, test_X_embedded = feature_selection1(train_X_scaled,train_y,test_X_scaled)
    #train_X_embedded, test_X_embedded = feature_selection2(train_X_scaled,train_y,test_X_scaled)

    grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)
    grid_search.fit(train_X, train_y) #,xgb_model__sample_weight=sample_weights) 

    best_pipeline = grid_search.best_estimator_
    best_params = grid_search.best_params_
    
    print(f"Best hyperparameters: {best_params}")
    wrf.write('  Best hyperparameters: {}\n'.format(best_params))
    #print(f"Best score (roc_auc): {grid_search.best_score_}")
    
    #pred_values=best_model.predict(test_X)
    pred_values = grid_search.predict(test_X)

    accr = accuracy_score(test_y,pred_values)
    accr_bal = balanced_accuracy_score(test_y,pred_values)
    f1s = f1_score(test_y,pred_values,average='weighted',zero_division = np.nan)
    prec = precision_score(test_y,pred_values,average='weighted',zero_division = np.nan)
    rec = recall_score(test_y,pred_values,average='weighted',zero_division = np.nan)
    #roc = roc_auc_score(test_y,pred_values,average='macro',multi_class='ovo')

    print(confusion_matrix(test_y, pred_values))
    wrf.write('  {}\n'.format(confusion_matrix(test_y, pred_values)))
    
    acc_scores.append(accr)
    acc_scores_bal.append(accr_bal)
    f1_scores.append(f1s)
    prec_scores.append(prec)
    rec_scores.append(rec)
    #roc_scores.append(roc)
    
    accr = None
    accr_bal = None 
    f1s = None 
    prec = None 
    rec = None 
    roc = None 
    #best_model = None 
    #best_params = None 


    ## === Feature importance === 

    col_names = np.array(X.columns)
    mask1 = best_pipeline.named_steps['variance'].get_support()
    mask2 = best_pipeline.named_steps['feature_sel1'].get_support()
    mask3 = best_pipeline.named_steps['feature_sel2'].get_support()
    mask4 = best_pipeline.named_steps['feature_sel3'].get_support()
    features_mask1 = col_names[mask1]
    features_mask2 = features_mask1[mask2]
    features_mask3 = features_mask2[mask3]
    feature_names = features_mask3[mask4]
    ## print("Selected Features:", feature_names)

    ## Native method
    print('\tNative importance')
    best_model = best_pipeline.named_steps['xgb_model']
    importance = best_model.feature_importances_
   
    #print(importance)
    for lk,feature in enumerate(feature_names): 
        if feature in nat_imp.keys(): 
            tmp = nat_imp[feature]
            tmp.append(importance[lk])
            nat_imp[feature] = tmp
            tmp = None 
        else:
            nat_imp[feature] = [importance[lk]]
            
    importance = None 
    lk = None 
    feature = None 

    ## Permutaiton method 
    print('\tPermutation importance')
    test_X_select = test_X[feature_names]
    result = permutation_importance(best_model, test_X_select.values, test_y, n_repeats=10, random_state=42)
    importance = result.importances_mean
    for lk,feature in enumerate(feature_names): 
        if feature in perm_imp.keys(): 
            tmp = perm_imp[feature]
            tmp.append(importance[lk])
            perm_imp[feature] = tmp
            tmp = None 
        else:
            perm_imp[feature] = [importance[lk]]    

    importance = None 
    lk = None 
    feature = None 
    result = None 

   ## SHAP 

    print('\tSHAP importance')
    explainer = shap.Explainer(best_model.predict_proba,test_X_select.values)
    shap_values = explainer(test_X_select.values)
    shap_df1 = pd.DataFrame(shap_values.values[:, :, 0], columns=test_X_select.columns)
    shap_df2 = pd.DataFrame(shap_values.values[:, :, 1], columns=test_X_select.columns)
 
    ## shap_df_sum = shap_df1.abs() + shap_df2.abs()
    
    importance_class0 = shap_df1.abs().mean()
    importance_class1 = shap_df2.abs().mean()
    
    ##print(type(importance))
    ##importance_df = pd.DataFrame({
    ##    'Feature': feature_names,
    ##    'Importance': shap_df_sum.abs().mean()
    ##})

    for lk,feature in enumerate(feature_names): 
        if feature in shap_imp.keys(): 
            tmp = shap_imp[feature]
            tmp.append([importance_class0.iloc[lk],importance_class1.iloc[lk]])
            shap_imp[feature] = tmp
            tmp = None 
        else:
            shap_imp[feature] = [[importance_class0.iloc[lk],importance_class1.iloc[lk]]]   

    shap_df1 = None 
    shap_df2 = None 
    shap_values = None 
    explainer = None 
    lk = None 
    feature = None 
    importance_class0 = None
    importance_class1 = None
   
    
    X = None 
    y = None 
    best_model = None 
    best_params = None
    train_X = None 
    test_X = None 
    train_y = None 
    test_y = None 
    test_X_select = None 



print(f'average_accuracy: {statistics.mean(acc_scores):.2f}, sd_accuracy: {statistics.pstdev(acc_scores):.2f}\n')
print(f'average_accuracy_balanced: {statistics.mean(acc_scores_bal):.2f}, sd_accuracy: {statistics.pstdev(acc_scores_bal):.2f}\n')
print(f'average_f1-score: {statistics.mean(f1_scores):.2f}, sd_f1-score: {statistics.pstdev(f1_scores):.2f}\n')
print(f'average_precision: {statistics.mean(prec_scores):.2f}, sd_precision: {statistics.pstdev(prec_scores):.2f}\n')
print(f'average_recall: {statistics.mean(rec_scores):.2f}, sd_Recall: {statistics.stdev(rec_scores):.2f}\n')
#print(f'average_ROC-AUC_score: {statistics.mean(roc_scores):.2f}, sd_ROC-AUC_score: {statistics.stdev(roc_scores):.2f}\n')
wrf.write('\taverage_accuracy: {}, sd_accuracy: {}\n'.format(round(statistics.mean(acc_scores),2), round(statistics.pstdev(acc_scores),2)))
wrf.write('\taverage_accuracy_balanced: {}, sd_accuracy_balanced: {}\n'.format(round(statistics.mean(acc_scores_bal),2), round(statistics.pstdev(acc_scores_bal),2)))
wrf.write('\taverage_f1-score: {}, sd_f1-score: {}\n'.format(round(statistics.mean(f1_scores),2), round(statistics.pstdev(f1_scores),2)))
wrf.write('\taverage_precision: {}, sd_precision: {}\n'.format(round(statistics.mean(prec_scores),2), round(statistics.pstdev(prec_scores),2)))
wrf.write('\taverage_recall: {}, sd_Recall: {}\n'.format(round(statistics.mean(rec_scores),2), round(statistics.pstdev(rec_scores),2)))

wrf.write("\n ===== Top Features by Native importance method =====\n")
print("Top Features by Native importance method:")

avg_imp = []
sd_imp = []

for key in nat_imp.keys():
    #print(key,nat_imp[key])
    if len(nat_imp[key]) < runs:
        xtlist = [0]*(runs-len(nat_imp[key]))
        tmp = nat_imp[key] + xtlist 
        nat_imp[key] = tmp 
        tmp = None 
        xtlist = None 
        
    #print(key,nat_imp[key])    
    avg = round(statistics.mean(nat_imp[key]),4)
    ##sd = round(statistics.stdev(nat_imp[key]),2)
    series = pd.Series(nat_imp[key]) 
    sd = round(series.std(ddof=0),4)
    series = None 

    avg_imp.append(avg)
    sd_imp.append(sd)

data = {
    'Features': nat_imp.keys(),
    'Mean_Imp': avg_imp, 
    'SD_Imp': sd_imp, 
}

feature_importance = pd.DataFrame(data)
feature_importance = feature_importance.sort_values(by='Mean_Imp', ascending=False)
print(feature_importance.head(20))
wrf.write('{}\n'.format(feature_importance))

feature_importance.to_csv('../results_Merged_Myl/LUAD_Myl_FeatureImp_XGB_Native.tsv',sep='\t',index=False)

avg_imp = None 
sd_imp = None 
nat_imp = None 
data = None 
feature_importance = None 


wrf.write("\n====== Top Features by Permutation Importance ======= \n")
print("\nTop Features by Permutation Importance:")

avg_imp = []
sd_imp = []

for key in perm_imp.keys():
    #print(key,nat_imp[key])
    if len(perm_imp[key]) < runs:
        xtlist = [0]*(runs-len(perm_imp[key]))
        tmp = perm_imp[key] + xtlist 
        perm_imp[key] = tmp 
        tmp = None 
        xtlist = None 
        
    #print(key,nat_imp[key])    
    avg = round(statistics.mean(perm_imp[key]),4)
    series = pd.Series(perm_imp[key]) 
    sd = round(series.std(ddof=0),4)
    series = None 
    
    avg_imp.append(avg)
    sd_imp.append(sd)

data = {
    'Features': perm_imp.keys(),
    'Mean_Imp': avg_imp, 
    'SD_Imp': sd_imp, 
}

feature_importance = pd.DataFrame(data)
feature_importance = feature_importance.sort_values(by='Mean_Imp', ascending=False)
print(feature_importance.head(20))
wrf.write('{}\n'.format(feature_importance))

feature_importance.to_csv('../results_Merged_Myl/LUAD_Myl_FeatureImp_XGB_Perm.tsv',sep='\t',index=False)

avg_imp = None 
sd_imp = None 
perm_imp = None 
data = None 
feature_importance = None 


print("Top Features by SHAP importance method:")

avg_imp = []
sd_imp = []

for key in shap_imp.keys():
   #print(key,shap_imp[key])
   if len(shap_imp[key]) < runs:
       xtlist = [[0]*numClasses]*(runs-len(shap_imp[key]))
       tmp = shap_imp[key] + xtlist 
       shap_imp[key] = tmp 
       tmp = None 
       xtlist = None 
        
   #print(key,shap_imp[key])    

   keyavg = []
   keysd = []
   for i in range(0,numClasses):
       vals = []
       for j in range(0,runs):
           #print(shap_imp[key][j][i])
           vals.append(shap_imp[key][j][i])

       avg = round(statistics.mean(vals),4)
       sd = round(statistics.stdev(vals),4)
       keyavg.append(avg)
       keysd.append(sd)
       avg = None 
       sd = None 
   ## print(keyavg,keysd)

   avg_imp.append(keyavg)
   sd_imp.append (keysd)



data = {
   'Features': shap_imp.keys(),
   'Mean_Imp': avg_imp, 
   'SD_Imp': sd_imp, 
}

feature_importance = pd.DataFrame(data)
feature_importance = feature_importance.sort_values(by='Mean_Imp', ascending=False)
print(feature_importance.head(20))

wrf.write('{}\n'.format(feature_importance))

feature_importance.to_csv('../results_Merged_Myl/LUAD_Myl_FeatureImp_XGB_SHAP.tsv',sep='\t',index=False)

avg_imp = None 
sd_imp = None 
shap_imp = None 
data = None 
feature_importance = None 

wrf.close()



Sampling data ...
0
status_encoded
0    543
1    543
Name: count, dtype: int64
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best hyperparameters: {'xgb_model__n_estimators': 50}
[[111  25]
 [ 31 105]]
	Native importance
	Permutation importance


Background dataset has 272 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=272 when initializing the masker.


	SHAP importance


PermutationExplainer explainer: 273it [00:17, 12.07it/s]                         


1
status_encoded
0    543
1    543
Name: count, dtype: int64
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best hyperparameters: {'xgb_model__n_estimators': 50}
[[118  18]
 [ 28 108]]
	Native importance
	Permutation importance


...

	SHAP importance


PermutationExplainer explainer: 273it [00:10,  1.04s/it]                         


4
status_encoded
0    543
1    543
Name: count, dtype: int64
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best hyperparameters: {'xgb_model__n_estimators': 150}
[[ 98  38]
 [ 27 109]]
	Native importance
	Permutation importance


Background dataset has 272 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=272 when initializing the masker.


	SHAP importance
average_accuracy: 0.79, sd_accuracy: 0.03

average_accuracy_balanced: 0.79, sd_accuracy: 0.03

average_f1-score: 0.79, sd_f1-score: 0.03

average_precision: 0.79, sd_precision: 0.03

average_recall: 0.79, sd_Recall: 0.03

Top Features by Native importance method:
         Features  Mean_Imp  SD_Imp
39         LGALS3    0.0321  0.0318
4            AQP9    0.0259  0.0149
168  RP11-598F7.3    0.0250  0.0500
27           FTH1    0.0235  0.0159
133       SLC11A1    0.0226  0.0359
51           MYL6    0.0204  0.0061
23          FABP4    0.0189  0.0111
232      SERPINA1    0.0156  0.0312
213          CES1    0.0139  0.0278
10       C12orf75    0.0129  0.0126
36         JCHAIN    0.0128  0.0101
138          XIST    0.0128  0.0255
67          RPS26    0.0125  0.0068
86            VIM    0.0124  0.0028
83           TSPO    0.0117  0.0069
172          SDC2    0.0115  0.0155
49            MGP    0.0102  0.0054
11          CCL18    0.0101  0.0078
3           APOC1    0.0101  0.0085

In [5]:
## ===== Random Forest ============= 

#clf = RandomForestClassifier(class_weight="balanced")
lrm = LogisticRegression(solver='lbfgs', C=0.1, max_iter=1000)
rfecv = RFECV(estimator=lrm, step=1, cv=StratifiedKFold(5), scoring='roc_auc')

pipeline = Pipeline(
    steps=[
        ("scaling",scaler),
        ("norm",normalizer),
        ("variance", VarianceThreshold()),
        ("feature_sel1",SelectKBest(score_func=f_classif, k=200)),
        ("feature_sel2",SelectFromModel(LogisticRegression(solver='lbfgs', C=0.1, max_iter=1000))),
        ("feature_sel3",rfecv),
        ("rf_model", RandomForestClassifier(n_estimators=100, max_depth=3))
    ]
)


param_grid = {
    "rf_model__max_depth": [ 4, 6, 8 ], 
    "rf_model__n_estimators": [ 50, 100, 150], 
}

wrname = '../results_Merged_Myl/Results_MLrun_Merged_LUAD_run_RF.txt'
wrf = open(wrname,'w')

acc_scores = []
acc_scores_bal = []
f1_scores = []
prec_scores = []
rec_scores = []
#roc_scores = []

nat_imp = {}
perm_imp = {}
shap_imp = {}


print('Sampling data ...')

for k in range(0,runs):
    print(k)
    wrf.write('Round: {}\n'.format(k))

    df1 = data_cleaned.loc[data_cleaned['status_encoded'] == 0]
    df2 = data_cleaned.loc[data_cleaned['status_encoded'] == 1]

    min_val = min(len(df1),len(df2))
  
    df1_sample = df1.sample(n=min_val)
    df1 = None 
    df2_sample = df2.sample(n=min_val)
    df2 = None 
 
    final_df = pd.concat([df1_sample, df2_sample], axis=0)
    df1_sample = None 
    df2_sample = None 

    
    print(final_df['status_encoded'].value_counts()) 

    xnames=final_df.columns[1:len(final_df.columns)-2]
    #print(xnames)
    X=final_df[xnames]
    y=final_df.status_encoded
    
    
    xnames = None 
    final_df = None  

    train_X, test_X, train_y, test_y = train_test_split(X, y, stratify=y, train_size=0.75)

    #sample_weights = compute_sample_weight(
    #    class_weight='balanced',
    #    y = train_y #provide your own target name
    #)
    
    grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)
    grid_search.fit(train_X, train_y) #,rf_model__sample_weight=sample_weights) 

    best_pipeline = grid_search.best_estimator_
    best_params = grid_search.best_params_
    
    print(f"Best hyperparameters: {best_params}")
    wrf.write('  Best hyperparameters: {}\n'.format(best_params))
    #print(f"Best score (roc_auc): {grid_search.best_score_}")
    
    #pred_values=best_model.predict(test_X)
    pred_values = grid_search.predict(test_X)

    accr = accuracy_score(test_y,pred_values)
    accr_bal = balanced_accuracy_score(test_y,pred_values)
    f1s = f1_score(test_y,pred_values,average='weighted',zero_division = np.nan)
    prec = precision_score(test_y,pred_values,average='weighted',zero_division = np.nan)
    rec = recall_score(test_y,pred_values,average='weighted',zero_division = np.nan)
    #roc = roc_auc_score(test_y,pred_values,average='macro',multi_class='ovo')

    print(confusion_matrix(test_y, pred_values))
    wrf.write('  {}\n'.format(confusion_matrix(test_y, pred_values)))
    
    acc_scores.append(accr)
    acc_scores_bal.append(accr_bal)
    f1_scores.append(f1s)
    prec_scores.append(prec)
    rec_scores.append(rec)
    #roc_scores.append(roc)
    
    accr = None
    accr_bal = None 
    f1s = None 
    prec = None 
    rec = None 
    roc = None 
    #best_model = None 
    #best_params = None 


    ## === Feature importance === 

    col_names = np.array(X.columns)
    mask1 = best_pipeline.named_steps['variance'].get_support()
    mask2 = best_pipeline.named_steps['feature_sel1'].get_support()
    mask3 = best_pipeline.named_steps['feature_sel2'].get_support()
    mask4 = best_pipeline.named_steps['feature_sel3'].get_support()
    features_mask1 = col_names[mask1]
    features_mask2 = features_mask1[mask2]
    features_mask3 = features_mask2[mask3]
    feature_names = features_mask3[mask4]
    ## print("Selected Features:", feature_names)

    ## Native method
    print('\tNative importance')
    best_model = best_pipeline.named_steps['rf_model']
    importance = best_model.feature_importances_
   
    #print(importance)
    for lk,feature in enumerate(feature_names): 
        if feature in nat_imp.keys(): 
            tmp = nat_imp[feature]
            tmp.append(importance[lk])
            nat_imp[feature] = tmp
            tmp = None 
        else:
            nat_imp[feature] = [importance[lk]]
            
    importance = None 
    lk = None 
    feature = None 

    ## Permutaiton method 
    print('\tPermutation importance')
    test_X_select = test_X[feature_names]
             
    result = permutation_importance(best_model, test_X_select.values, test_y, n_repeats=10, random_state=42)
    importance = result.importances_mean
    for lk,feature in enumerate(feature_names): 
        if feature in perm_imp.keys(): 
            tmp = perm_imp[feature]
            tmp.append(importance[lk])
            perm_imp[feature] = tmp
            tmp = None 
        else:
            perm_imp[feature] = [importance[lk]]    

    importance = None 
    lk = None 
    feature = None 
    result = None 

   ## SHAP 

    print('\tSHAP importance')
    explainer = shap.Explainer(best_model.predict_proba,test_X_select.values)
    shap_values = explainer(test_X_select.values)
    shap_df1 = pd.DataFrame(shap_values.values[:, :, 0], columns=test_X_select.columns)
    shap_df2 = pd.DataFrame(shap_values.values[:, :, 1], columns=test_X_select.columns)
 
    ## shap_df_sum = shap_df1.abs() + shap_df2.abs()
    
    importance_class0 = shap_df1.abs().mean()
    importance_class1 = shap_df2.abs().mean()
    
    ##print(type(importance))
    ##importance_df = pd.DataFrame({
    ##    'Feature': feature_names,
    ##    'Importance': shap_df_sum.abs().mean()
    ##})

    for lk,feature in enumerate(feature_names): 
        if feature in shap_imp.keys(): 
            tmp = shap_imp[feature]
            tmp.append([importance_class0.iloc[lk],importance_class1.iloc[lk]])
            shap_imp[feature] = tmp
            tmp = None 
        else:
            shap_imp[feature] = [[importance_class0.iloc[lk],importance_class1.iloc[lk]]]   

    shap_df1 = None 
    shap_df2 = None 
    shap_values = None 
    explainer = None 
    lk = None 
    feature = None 
    importance_class0 = None
    importance_class1 = None
   
    
    X = None 
    y = None 
    best_model = None 
    best_params = None
    train_X = None 
    test_X = None 
    train_y = None 
    test_y = None 
    test_X_select = None 



print(f'average_accuracy: {statistics.mean(acc_scores):.2f}, sd_accuracy: {statistics.pstdev(acc_scores):.2f}\n')
print(f'average_accuracy_balanced: {statistics.mean(acc_scores_bal):.2f}, sd_accuracy: {statistics.pstdev(acc_scores_bal):.2f}\n')
print(f'average_f1-score: {statistics.mean(f1_scores):.2f}, sd_f1-score: {statistics.pstdev(f1_scores):.2f}\n')
print(f'average_precision: {statistics.mean(prec_scores):.2f}, sd_precision: {statistics.pstdev(prec_scores):.2f}\n')
print(f'average_recall: {statistics.mean(rec_scores):.2f}, sd_Recall: {statistics.stdev(rec_scores):.2f}\n')
#print(f'average_ROC-AUC_score: {statistics.mean(roc_scores):.2f}, sd_ROC-AUC_score: {statistics.stdev(roc_scores):.2f}\n')
wrf.write('\taverage_accuracy: {}, sd_accuracy: {}\n'.format(round(statistics.mean(acc_scores),2), round(statistics.pstdev(acc_scores),2)))
wrf.write('\taverage_accuracy_balanced: {}, sd_accuracy_balanced: {}\n'.format(round(statistics.mean(acc_scores_bal),2), round(statistics.pstdev(acc_scores_bal),2)))
wrf.write('\taverage_f1-score: {}, sd_f1-score: {}\n'.format(round(statistics.mean(f1_scores),2), round(statistics.pstdev(f1_scores),2)))
wrf.write('\taverage_precision: {}, sd_precision: {}\n'.format(round(statistics.mean(prec_scores),2), round(statistics.pstdev(prec_scores),2)))
wrf.write('\taverage_recall: {}, sd_Recall: {}\n'.format(round(statistics.mean(rec_scores),2), round(statistics.pstdev(rec_scores),2)))

wrf.write("\n ===== Top Features by Native importance method =====\n")
print("Top Features by Native importance method:")

avg_imp = []
sd_imp = []

for key in nat_imp.keys():
    #print(key,nat_imp[key])
    if len(nat_imp[key]) < runs:
        xtlist = [0]*(runs-len(nat_imp[key]))
        tmp = nat_imp[key] + xtlist 
        nat_imp[key] = tmp 
        tmp = None 
        xtlist = None 
        
    #print(key,nat_imp[key])    
    avg = round(statistics.mean(nat_imp[key]),4)
    ##sd = round(statistics.stdev(nat_imp[key]),2)
    series = pd.Series(nat_imp[key]) 
    sd = round(series.std(ddof=0),4)
    series = None 

    avg_imp.append(avg)
    sd_imp.append(sd)

data = {
    'Features': nat_imp.keys(),
    'Mean_Imp': avg_imp, 
    'SD_Imp': sd_imp, 
}

feature_importance = pd.DataFrame(data)
feature_importance = feature_importance.sort_values(by='Mean_Imp', ascending=False)
print(feature_importance.head(20))
wrf.write('{}\n'.format(feature_importance))

feature_importance.to_csv('../results_Merged_Myl/LUAD_Myl_FeatureImp_RF_Native.tsv',sep='\t',index=False)

avg_imp = None 
sd_imp = None 
nat_imp = None 
data = None 
feature_importance = None 


wrf.write("\n====== Top Features by Permutation Importance ======= \n")
print("\nTop Features by Permutation Importance:")

avg_imp = []
sd_imp = []

for key in perm_imp.keys():
    #print(key,nat_imp[key])
    if len(perm_imp[key]) < runs:
        xtlist = [0]*(runs-len(perm_imp[key]))
        tmp = perm_imp[key] + xtlist 
        perm_imp[key] = tmp 
        tmp = None 
        xtlist = None 
        
    #print(key,nat_imp[key])    
    avg = round(statistics.mean(perm_imp[key]),4)
    series = pd.Series(perm_imp[key]) 
    sd = round(series.std(ddof=0),4)
    series = None 
    
    avg_imp.append(avg)
    sd_imp.append(sd)

data = {
    'Features': perm_imp.keys(),
    'Mean_Imp': avg_imp, 
    'SD_Imp': sd_imp, 
}

feature_importance = pd.DataFrame(data)
feature_importance = feature_importance.sort_values(by='Mean_Imp', ascending=False)
print(feature_importance.head(20))
wrf.write('{}\n'.format(feature_importance))

feature_importance.to_csv('../results_Merged_Myl/LUAD_Myl_FeatureImp_RF_Perm.tsv',sep='\t',index=False)

avg_imp = None 
sd_imp = None 
perm_imp = None 
data = None 
feature_importance = None 


print("Top Features by SHAP importance method:")

avg_imp = []
sd_imp = []

for key in shap_imp.keys():
   #print(key,shap_imp[key])
   if len(shap_imp[key]) < runs:
       xtlist = [[0]*numClasses]*(runs-len(shap_imp[key]))
       tmp = shap_imp[key] + xtlist 
       shap_imp[key] = tmp 
       tmp = None 
       xtlist = None 
        
   #print(key,shap_imp[key])    

   keyavg = []
   keysd = []
   for i in range(0,numClasses):
       vals = []
       for j in range(0,runs):
           #print(shap_imp[key][j][i])
           vals.append(shap_imp[key][j][i])

       avg = round(statistics.mean(vals),4)
       sd = round(statistics.stdev(vals),4)
       keyavg.append(avg)
       keysd.append(sd)
       avg = None 
       sd = None 
   ## print(keyavg,keysd)

   avg_imp.append(keyavg)
   sd_imp.append (keysd)



data = {
   'Features': shap_imp.keys(),
   'Mean_Imp': avg_imp, 
   'SD_Imp': sd_imp, 
}

feature_importance = pd.DataFrame(data)
feature_importance = feature_importance.sort_values(by='Mean_Imp', ascending=False)
print(feature_importance.head(20))

wrf.write('{}\n'.format(feature_importance))

feature_importance.to_csv('../results_Merged_Myl/LUAD_Myl_FeatureImp_RF_SHAP.tsv',sep='\t',index=False)

avg_imp = None 
sd_imp = None 
shap_imp = None 
data = None 
feature_importance = None 

wrf.close()




Sampling data ...
0
status_encoded
0    543
1    543
Name: count, dtype: int64
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best hyperparameters: {'rf_model__max_depth': 8, 'rf_model__n_estimators': 150}
[[ 94  42]
 [ 26 110]]
	Native importance
	Permutation importance


Background dataset has 272 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=272 when initializing the masker.


	SHAP importance


PermutationExplainer explainer: 273it [00:49,  4.35it/s]                         


1
status_encoded
0    543
1    543
Name: count, dtype: int64
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best hyperparameters: {'rf_model__max_depth': 6, 'rf_model__n_estimators': 150}
[[107  29]
 [ 29 107]]
	Native importance
	Permutation importance


...

4
status_encoded
0    543
1    543
Name: count, dtype: int64
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best hyperparameters: {'rf_model__max_depth': 8, 'rf_model__n_estimators': 100}
[[109  27]
 [ 32 104]]
	Native importance
	Permutation importance


Background dataset has 272 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=272 when initializing the masker.


	SHAP importance


PermutationExplainer explainer: 273it [00:40,  4.98it/s]                         


average_accuracy: 0.77, sd_accuracy: 0.02

average_accuracy_balanced: 0.77, sd_accuracy: 0.02

average_f1-score: 0.77, sd_f1-score: 0.02

average_precision: 0.77, sd_precision: 0.02

average_recall: 0.77, sd_Recall: 0.02

Top Features by Native importance method:
    Features  Mean_Imp  SD_Imp
26      FTH1    0.0454  0.0055
66     RPS26    0.0332  0.0061
48      MYL6    0.0293  0.0075
152   LGALS3    0.0247  0.0202
27       FTL    0.0245  0.0164
31     GSTO1    0.0196  0.0171
93      CSTB    0.0192  0.0174
80     TREM1    0.0187  0.0107
79     TIMP1    0.0169  0.0094
110     IGKC    0.0157  0.0138
81      TSPO    0.0156  0.0089
78     TCEB2    0.0144  0.0081
15     CEBPB    0.0143  0.0017
61    RPL23A    0.0137  0.0094
129   S100A6    0.0129  0.0106
168  S100A11    0.0126  0.0160
127    RPS21    0.0125  0.0106
137      VIM    0.0119  0.0061
121     OAZ1    0.0114  0.0142
42     MARCO    0.0108  0.0033

Top Features by Permutation Importance:
    Features  Mean_Imp  SD_Imp
110     IGKC 

In [47]:
## ======= Logistic regression ==========

lr_model=LogisticRegression(solver='saga',max_iter=10000)

#clf = RandomForestClassifier(class_weight="balanced")
lrm = LogisticRegression(solver='lbfgs', C=0.1, max_iter=1000)
rfecv = RFECV(estimator=lrm, step=1, cv=StratifiedKFold(5), scoring='roc_auc')

pipeline = Pipeline(
    steps=[
        ("scaling",scaler),
        ("norm",normalizer),
        ("variance", VarianceThreshold()),
        ("feature_sel1",SelectKBest(score_func=f_classif, k=200)),
        ("feature_sel2",SelectFromModel(LogisticRegression(solver='lbfgs', C=0.1, max_iter=1000))),
        ("feature_sel3",rfecv),
        ("lr_model", LogisticRegression(solver='saga',max_iter=10000))
    ]
)

param_grid = {
    "lr_model__C": [ 0.1, 0.5, 1], 
    "lr_model__l1_ratio": [ 0, 0.5, 1]
}

wrname = '../results_Merged_Myl/Results_MLrun_Merged_LUAD_run_LogReg.txt'
wrf = open(wrname,'w')

acc_scores = []
acc_scores_bal = []
f1_scores = []
prec_scores = []
rec_scores = []
#roc_scores = []

nat_imp = {}
perm_imp = {}
shap_imp = {}


print('Sampling data ...')

for k in range(0,runs):
    print(k)
    wrf.write('Round: {}\n'.format(k))

    df1 = data_cleaned.loc[data_cleaned['status_encoded'] == 0]
    df2 = data_cleaned.loc[data_cleaned['status_encoded'] == 1]

    min_val = min(len(df1),len(df2))
  
    df1_sample = df1.sample(n=min_val)
    df1 = None 
    df2_sample = df2.sample(n=min_val)
    df2 = None 
 
    final_df = pd.concat([df1_sample, df2_sample], axis=0)
    df1_sample = None 
    df2_sample = None 

    
    print(final_df['status_encoded'].value_counts()) 

    xnames=final_df.columns[1:len(final_df.columns)-2]
    #print(xnames)
    X=final_df[xnames]
    y=final_df.status_encoded
    
    xnames = None 
    final_df = None  

    train_X, test_X, train_y, test_y = train_test_split(X, y, stratify=y, train_size=0.75)

    #sample_weights = compute_sample_weight(
    #    class_weight='balanced',
    #    y = train_y #provide your own target name
    #)

    #train_X_embedded, test_X_embedded = feature_selection1(train_X_scaled,train_y,test_X_scaled)
    #train_X_embedded, test_X_embedded = feature_selection2(train_X_scaled,train_y,test_X_scaled)

    grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)
    grid_search.fit(train_X, train_y) #,xgb_model__sample_weight=sample_weights) 

    best_pipeline = grid_search.best_estimator_
    best_params = grid_search.best_params_
    
    print(f"Best hyperparameters: {best_params}")
    wrf.write('  Best hyperparameters: {}\n'.format(best_params))
    #print(f"Best score (roc_auc): {grid_search.best_score_}")
    
    #pred_values=best_model.predict(test_X)
    pred_values = grid_search.predict(test_X)

    accr = accuracy_score(test_y,pred_values)
    accr_bal = balanced_accuracy_score(test_y,pred_values)
    f1s = f1_score(test_y,pred_values,average='weighted',zero_division = np.nan)
    prec = precision_score(test_y,pred_values,average='weighted',zero_division = np.nan)
    rec = recall_score(test_y,pred_values,average='weighted',zero_division = np.nan)
    #roc = roc_auc_score(test_y,pred_values,average='macro',multi_class='ovo')

    print(confusion_matrix(test_y, pred_values))
    wrf.write('  {}\n'.format(confusion_matrix(test_y, pred_values)))
    
    acc_scores.append(accr)
    acc_scores_bal.append(accr_bal)
    f1_scores.append(f1s)
    prec_scores.append(prec)
    rec_scores.append(rec)
    #roc_scores.append(roc)
    
    accr = None
    accr_bal = None 
    f1s = None 
    prec = None 
    rec = None 
    roc = None 
    #best_model = None 
    #best_params = None 


    ## === Feature importance === 

    best_model = best_pipeline.named_steps['lr_model']

    col_names = np.array(X.columns)
    mask1 = best_pipeline.named_steps['variance'].get_support()
    mask2 = best_pipeline.named_steps['feature_sel1'].get_support()
    mask3 = best_pipeline.named_steps['feature_sel2'].get_support()
    mask4 = best_pipeline.named_steps['feature_sel3'].get_support()
    features_mask1 = col_names[mask1]
    features_mask2 = features_mask1[mask2]
    features_mask3 = features_mask2[mask3]
    feature_names = features_mask3[mask4]
    ## print("Selected Features:", feature_names)

    ## Permutaiton method 
    print('\tPermutation importance')
    test_X_select = test_X[feature_names]
    result = permutation_importance(best_model, test_X_select.values, test_y, n_repeats=10, random_state=42)
    importance = result.importances_mean
    for lk,feature in enumerate(feature_names): 
        if feature in perm_imp.keys(): 
            tmp = perm_imp[feature]
            tmp.append(importance[lk])
            perm_imp[feature] = tmp
            tmp = None 
        else:
            perm_imp[feature] = [importance[lk]]    

    importance = None 
    lk = None 
    feature = None 
    result = None 
  
    X = None 
    y = None 
    best_model = None 
    best_params = None
    train_X = None 
    test_X = None 
    train_y = None 
    test_y = None 
    test_X_select = None 



print(f'average_accuracy: {statistics.mean(acc_scores):.2f}, sd_accuracy: {statistics.pstdev(acc_scores):.2f}\n')
print(f'average_accuracy_balanced: {statistics.mean(acc_scores_bal):.2f}, sd_accuracy: {statistics.pstdev(acc_scores_bal):.2f}\n')
print(f'average_f1-score: {statistics.mean(f1_scores):.2f}, sd_f1-score: {statistics.pstdev(f1_scores):.2f}\n')
print(f'average_precision: {statistics.mean(prec_scores):.2f}, sd_precision: {statistics.pstdev(prec_scores):.2f}\n')
print(f'average_recall: {statistics.mean(rec_scores):.2f}, sd_Recall: {statistics.stdev(rec_scores):.2f}\n')
#print(f'average_ROC-AUC_score: {statistics.mean(roc_scores):.2f}, sd_ROC-AUC_score: {statistics.stdev(roc_scores):.2f}\n')
wrf.write('\taverage_accuracy: {}, sd_accuracy: {}\n'.format(round(statistics.mean(acc_scores),2), round(statistics.pstdev(acc_scores),2)))
wrf.write('\taverage_accuracy_balanced: {}, sd_accuracy_balanced: {}\n'.format(round(statistics.mean(acc_scores_bal),2), round(statistics.pstdev(acc_scores_bal),2)))
wrf.write('\taverage_f1-score: {}, sd_f1-score: {}\n'.format(round(statistics.mean(f1_scores),2), round(statistics.pstdev(f1_scores),2)))
wrf.write('\taverage_precision: {}, sd_precision: {}\n'.format(round(statistics.mean(prec_scores),2), round(statistics.pstdev(prec_scores),2)))
wrf.write('\taverage_recall: {}, sd_Recall: {}\n'.format(round(statistics.mean(rec_scores),2), round(statistics.pstdev(rec_scores),2)))

wrf.write("\n====== Top Features by Permutation Importance ======= \n")
print("\nTop Features by Permutation Importance:")

avg_imp = []
sd_imp = []

for key in perm_imp.keys():
    #print(key,nat_imp[key])
    if len(perm_imp[key]) < runs:
        xtlist = [0]*(runs-len(perm_imp[key]))
        tmp = perm_imp[key] + xtlist 
        perm_imp[key] = tmp 
        tmp = None 
        xtlist = None 
        
    #print(key,nat_imp[key])    
    avg = round(statistics.mean(perm_imp[key]),4)
    series = pd.Series(perm_imp[key]) 
    sd = round(series.std(ddof=0),4)
    series = None 
    
    avg_imp.append(avg)
    sd_imp.append(sd)

data = {
    'Features': perm_imp.keys(),
    'Mean_Imp': avg_imp, 
    'SD_Imp': sd_imp, 
}

feature_importance = pd.DataFrame(data)
feature_importance = feature_importance.sort_values(by='Mean_Imp', ascending=False)
print(feature_importance.head(20))
wrf.write('{}\n'.format(feature_importance))

feature_importance.to_csv('../results_Merged_Myl/LUAD_Myl_FeatureImp_LogReg_Perm.tsv',sep='\t',index=False)

avg_imp = None 
sd_imp = None 
perm_imp = None 
data = None 
feature_importance = None 

wrf.close()


Sampling data ...
0
status_encoded
0    300
1    300
Name: count, dtype: int64
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best hyperparameters: {'lr_model__C': 0.1, 'lr_model__l1_ratio': 1}
[[53 22]
 [27 48]]
	Permutation importance
1
status_encoded
0    300
1    300
Name: count, dtype: int64
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best hyperparameters: {'lr_model__C': 0.1, 'lr_model__l1_ratio': 0.5}
[[49 26]
 [23 52]]
	Permutation importance
average_accuracy: 0.67, sd_accuracy: 0.00

average_accuracy_balanced: 0.67, sd_accuracy: 0.00

average_f1-score: 0.67, sd_f1-score: 0.00

average_precision: 0.67, sd_precision: 0.00

average_recall: 0.67, sd_Recall: 0.00


Top Features by Permutation Importance:
    Features  Mean_Imp  SD_Imp
84      CRYM    0.0073  0.0073
116    NR3C1    0.0057  0.0057
4       C1QC    0.0037  0.0037
44       LTB    0.0033  0.0033
100     IFI6    0.0027  0.0027
19     EGLN3    0.0023  0.0023
65     STAB1    0.0020  0.0020
47   

In [22]:
data_cleaned = None 